# PyHMMER

![PyHMMER](https://proto-bio.github.io/proto-assets/images/tool/pyhmmer/hero.png)

This notebook demonstrates the five PyHMMER tools. `run_pyhmmer_hmmsearch` searches HMM profiles against sequences to find family members. `run_pyhmmer_hmmscan` scans sequences against a profile database to annotate domain content. `run_pyhmmer_phmmer` performs protein-versus-protein search by building a temporary HMM on the fly. `run_pyhmmer_nhmmer` does the same for nucleotide sequences. `run_pyhmmer_jackhmmer` extends phmmer with iterative refinement to detect very distant homologs.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("pyhmmer")
display_overview("pyhmmer")
display_docs_section("pyhmmer", "Background")

# PyHMMER

[PyHMMER](https://github.com/althonos/pyhmmer) is a Python library that binds [HMMER3](http://hmmer.org/) for [profile hidden Markov model](https://en.wikipedia.org/wiki/Hidden_Markov_model) sequence search and domain annotation. It exposes the five canonical HMMER programs (`hmmsearch`, `hmmscan`, `phmmer`, `nhmmer`, `jackhmmer`) as Python functions, returns structured per-hit and per-domain results, and reaches the sensitivity of HMMER while staying entirely in-process.

[PyHMMER](https://github.com/althonos/pyhmmer) ([Larralde & Zeller, 2023](https://doi.org/10.1093/bioinformatics/btad214)) is a Cython binding to the HMMER C API that ships the HMMER source itself, so a single `pip install` provides both the Python interface and the compiled search engine. The underlying [HMMER3](http://hmmer.org/) algorithm ([Eddy, 2011](https://doi.org/10.1371/journal.pcbi.1002195)) builds a profile hidden Markov model from a [multiple sequence alignment](https://en.wikipedia.org/wiki/Multiple_sequence_alignment), where each match state stores position-specific emission probabilities and the transitions between states model insertions and deletions. Search proceeds through a cascade of accelerated filters: a [SIMD](https://en.wikipedia.org/wiki/Single_instruction,_multiple_data)-vectorised multiple-segment Viterbi (MSV) filter, a vectorised Viterbi filter, and a Forward/Backward filter, each tightening the candidate set before the final scored alignment. Each hit carries a database-size-independent bit score together with an [E-value](https://en.wikipedia.org/wiki/E-value) derived from extreme-value-distribution theory. The E-value calibrates the expected number of false-positive hits at that bit score for the database being searched.

Profile HMMs detect homology that pairwise methods such as [BLAST](https://en.wikipedia.org/wiki/BLAST_(biotechnology)) miss because they encode an entire family's position-specific conservation pattern rather than the similarity of two sequences alone. HMMER3 brought profile-HMM search within roughly the runtime envelope of BLAST while keeping that sensitivity advantage. PyHMMER preserves the algorithm exactly and adds Python-native multithreading, in-memory HMM and sequence handles, and structured result objects. Coordinates returned for HMM matches, target alignments, and envelopes are reported as 1-indexed, inclusive intervals to match biological residue selection conventions.

## Available tools

In [2]:
display_available_tools("pyhmmer")

- **`run_pyhmmer_hmmscan()`** — Search sequences against HMM database using PyHMMER
- **`run_pyhmmer_hmmsearch()`** — Search HMM profile(s) against sequences using PyHMMER
- **`run_pyhmmer_jackhmmer()`** — Iteratively search protein sequences against protein database using PyHMMER
- **`run_pyhmmer_nhmmer()`** — Search nucleotide sequences against nucleotide database using PyHMMER
- **`run_pyhmmer_phmmer()`** — Search protein sequences against protein database using PyHMMER

### `run_pyhmmer_hmmsearch`

`run_pyhmmer_hmmsearch` takes one or more HMM profiles and searches them against a set of protein sequences to identify family members. This is the standard approach when you already have a curated profile for a protein family and want to scan a proteome or custom sequence set for matches. Results include per-sequence and per-domain hits with E-values and bit scores.

In [3]:
from pathlib import Path

import proto_tools
from proto_tools import (
    PyHmmsearchInput,
    PyHmmsearchConfig,
    run_pyhmmer_hmmsearch,
)

In [4]:
# Display input docs
display_api_reference("pyhmmer", "input", "run_pyhmmer_hmmsearch")

# Path to HMM file containing domain profiles (Pkinase, Barwin, IPK, 1-cysPrx_C)
_repo_root = Path(proto_tools.__file__).resolve().parent.parent
hmm_path = _repo_root / "tests" / "dummy_data" / "test_multiple_hmm.hmm"

# Target proteins: an inositol polyphosphate kinase (matches the IPK + Barwin profiles)
# and hemoglobin alpha (an unrelated negative control)
kinase_seq = (
    "MLEPLPCWDAAKDLKEPQCPPGDRVGVQPGNSRVWQGTMEKAGLAWTRGTGVQSEGTWES"
    "QRQDSDALPSPELLPQDQDKPFLRKACSPSNIPAVIITDMGTQEDGALEETQGSPRGNLP"
    "LRKLSSSSASSTGFSSSYEDSEEDISSDPERTLDPNSAFLHTLDQQKPRVSKSWRKIKNM"
    "VHWSPFVMSFKKKYPWIQLAGHAGSFKAAANGRILKKHCESEQRCLDRLMVDVLRPFVPA"
    "YHGDVVKDGERYNQMDDLLADFDSPCVMDCKMGIRTYLEEELTKARRKPSLRKDMYQKMI"
    "EVDPEAPTEEEKAQRAVTKPRYMQWRETISSTATLGFRIEGIKKEDGTVNRDFKKTKTRE"
    "QVTEAFREFTKGNHNILIAYRDRLKAIRTTLEVSPFFKCHEVIGSSLLFIHDKKEQAKVW"
    "MIDFGKTTPLPEGQTLQHDVPWQEGNREDGYLSGLNNLVDILTEMSQDAPLA"
)

hemoglobin_seq = (
    "MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSH"
    "GSAQVKGHGKKVADALTNAVAHVDDMPNALSALSDLHAHKLRVDPVNFKL"
    "LSHCLLVTLAAHLPAEFTPAVHASLDKFLASVSTVLTSKYR"
)

inputs = PyHmmsearchInput(
    hmm=str(hmm_path),
    sequences=[kinase_seq, hemoglobin_seq],
)

**Input** — `PyHmmsearchInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequences</code> | <code>list[str]</code> | required | Input sequences as: single sequence string or list of sequence strings |
| <code>hmm</code> | <code>str &#124; Path</code> | required | Path to HMM file |

In [5]:
# Display config docs
display_api_reference("pyhmmer", "config", "run_pyhmmer_hmmsearch")

# Report all hits below E-value 10.0 | see docs above for all fields
config = PyHmmsearchConfig(evalue_threshold=10.0)

**Config** — `PyHmmsearchConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>verbose</code> | <code>int</code> | <code>0</code> | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| <code>device</code> | <code>str</code> | <code>'cpu'</code> | Device to run the tool on (e.g., 'cpu', 'cuda', 'cuda:0', 'cloud') |
| <code>timeout</code> | <code>int &#124; None</code> | <code>3600</code> | Maximum execution time in seconds. None waits indefinitely. |
| <code>seed</code> | <code>int &#124; None</code> | <code>None</code> | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| <code>num_threads</code> | <code>int</code> | <code>0</code> | CPU threads. 0 auto-detects available cores |
| <code>evalue_threshold</code> | <code>float</code> | <code>10.0</code> | Sequence E-value cap. Lower = stricter; 0.001 for confident, 1e-10 for stringent |
| <code>score_threshold</code> | <code>float &#124; None</code> | <code>None</code> | Sequence bit-score floor. Use for cross-DB-size comparisons; overrides E-value if set |
| <code>domain_evalue_threshold</code> | <code>float</code> | <code>10.0</code> | Domain E-value cap. Tighten independently from sequence threshold |
| <code>domain_score_threshold</code> | <code>float &#124; None</code> | <code>None</code> | Domain bit-score floor. For cross-DB-size comparisons; overrides domain E-value |
| <code>inclusion_evalue_threshold</code> | <code>float</code> | <code>0.01</code> | Inclusion E-value cap. Sets 'included' flag; seeds jackhmmer next iteration |
| <code>inclusion_domain_evalue_threshold</code> | <code>float</code> | <code>0.01</code> | Domain inclusion E-value cap. Sets included flag on domain hits |
| <code>z_value</code> | <code>float &#124; None</code> | <code>None</code> | Effective DB size for E-value calc. Set constant for cross-DB compare; None = actual count |
| <code>domain_z_value</code> | <code>float &#124; None</code> | <code>None</code> | Significant hits for domain E-value calc. Set with Z for cross-DB; None = actual |
| <code>skip_filters</code> | <code>bool</code> | <code>False</code> | Disable MSV/Vit/Fwd + bias filters. 10-100x slower but max sensitivity for distant homologs |
| <code>bit_cutoffs</code> | <code>Literal['gathering', 'noise', 'trusted'] &#124; None</code> | <code>None</code> | HMM curated cutoff: 'gathering' (Pfam GA), 'noise' (permissive), 'trusted' (strictest) |

In [6]:
# Run HMM profile search
hmmsearch_result = run_pyhmmer_hmmsearch(inputs, config)

Running run_pyhmmer_hmmsearch [00:00]

In [7]:
# Display output docs
display_api_reference("pyhmmer", "output", "run_pyhmmer_hmmsearch")

# The kinase (seq 0) should match IPK and Barwin profiles; hemoglobin (seq 1) should not
print(f"Found {hmmsearch_result.num_sequence_hits} sequence hits")
print(f"Found {hmmsearch_result.num_domain_hits} domain hits")

if hmmsearch_result.sequence_hits:
    print("\nSequence hits (target 0=kinase, 1=hemoglobin):")
    for hit in hmmsearch_result.sequence_hits:
        print(f"  query={hit.query_name}, target={hit.target_name}, evalue={hit.evalue:.2e}, score={hit.score:.1f}")

**Output** — `PyHmmerOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequence_hits</code> | <code>list[SequenceHit]</code> | <code>[]</code> | Sequence-level hits from the search |
| <code>domain_hits</code> | <code>list[DomainHit]</code> | <code>[]</code> | Domain-level hits from the search |

**`SequenceHit`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>query_name</code> | <code>str</code> | required | Name of the query HMM |
| <code>query_accession</code> | <code>str</code> | required | Accession of the query HMM. If not available, set to '-'. |
| <code>query_description</code> | <code>str</code> | required | Description of the query HMM. If not available, set to '-'. |
| <code>query_idx</code> | <code>int</code> | required | Index of the query (0-indexed) |
| <code>target_name</code> | <code>str</code> | required | Name of the target sequence |
| <code>target_accession</code> | <code>str</code> | required | Accession of the target sequence. If not available, set to '-'. |
| <code>target_description</code> | <code>str</code> | required | Description of the target sequence. If not available, set to '-'. |
| <code>evalue</code> | <code>float</code> | required | E-value of the hit |
| <code>score</code> | <code>float</code> | required | Bit score of the full sequence |
| <code>bias</code> | <code>float</code> | required | Bias correction for the sequence score |
| <code>sum_score</code> | <code>float</code> | required | Sum of domain scores |
| <code>reported</code> | <code>bool</code> | required | Whether the hit is reported |
| <code>included</code> | <code>bool</code> | required | Whether the hit is included |
| <code>pvalue</code> | <code>float</code> | required | p-value of the hit |
| <code>num_domains</code> | <code>int</code> | required | Number of domains in the hit |

**`DomainHit`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>query_name</code> | <code>str</code> | required | Name of the query HMM |
| <code>query_accession</code> | <code>str</code> | required | Accession of the query HMM. If not available, set to '-'. |
| <code>query_description</code> | <code>str</code> | required | Description of the query HMM. If not available, set to '-'. |
| <code>query_idx</code> | <code>int</code> | required | Index of the query (0-indexed) |
| <code>target_name</code> | <code>str</code> | required | Name of the target sequence |
| <code>target_accession</code> | <code>str</code> | required | Accession of the target sequence. If not available, set to '-'. |
| <code>target_description</code> | <code>str</code> | required | Description of the target sequence. If not available, set to '-'. |
| <code>hmm_length</code> | <code>int</code> | required | Length of the HMM profile |
| <code>hmm_from</code> | <code>int</code> | required | Start position of the domain match in the HMM (1-indexed) |
| <code>hmm_to</code> | <code>int</code> | required | End position of the domain match in the HMM (1-indexed) |
| <code>target_from</code> | <code>int</code> | required | Start position of the domain match in the target (1-indexed) |
| <code>target_to</code> | <code>int</code> | required | End position of the domain match in the target (1-indexed) |
| <code>target_length</code> | <code>int</code> | required | Length of the target sequence |
| <code>c_evalue</code> | <code>float</code> | required | Conditional E-value of the domain |
| <code>i_evalue</code> | <code>float</code> | required | Independent E-value of the domain |
| <code>domain_score</code> | <code>float</code> | required | Bit score of the domain |
| <code>domain_bias</code> | <code>float</code> | required | Bias correction of the domain score |
| <code>domain_idx</code> | <code>int</code> | required | Index of the domain within the hit (0-indexed) |
| <code>env_from</code> | <code>int</code> | required | Start position of the domain envelope in the target (1-indexed) |
| <code>env_to</code> | <code>int</code> | required | End position of the domain envelope in the target (1-indexed) |
| <code>envelope_score</code> | <code>float</code> | required | Bit score of the domain envelope |
| <code>domain_included</code> | <code>bool</code> | required | Whether the domain passes inclusion thresholds |
| <code>domain_reported</code> | <code>bool</code> | required | Whether the domain passes reporting thresholds |
| <code>domain_pvalue</code> | <code>float</code> | required | P-value of the domain |

Found 2 sequence hits
Found 2 domain hits

Sequence hits (target 0=kinase, 1=hemoglobin):
  query=Barwin, target=0, evalue=2.08e-05, score=11.7
  query=IPK, target=0, evalue=6.50e-50, score=156.8


### `run_pyhmmer_hmmscan`

`run_pyhmmer_hmmscan` is the reverse of hmmsearch: given query sequences, it scans them against a database of HMM profiles to annotate which domains are present. This is the standard workflow for domain architecture analysis, answering "what domains does this protein contain?" Results include per-sequence and per-domain hits with conditional E-values relative to the full profile database.

In [8]:
from proto_tools import (
    PyHmmscanInput,
    PyHmmscanConfig,
    run_pyhmmer_hmmscan,
)

In [9]:
# Display input docs
display_api_reference("pyhmmer", "input", "run_pyhmmer_hmmscan")

# Scan the same kinase and hemoglobin sequences against the HMM database
inputs = PyHmmscanInput(
    hmm_db=str(hmm_path),
    sequences=[kinase_seq, hemoglobin_seq],
)

**Input** — `PyHmmscanInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequences</code> | <code>list[str]</code> | required | Input sequences as: single sequence string or list of sequence strings |
| <code>hmm_db</code> | <code>str &#124; Path</code> | required | Path to HMM database file |

In [10]:
# Display config docs
display_api_reference("pyhmmer", "config", "run_pyhmmer_hmmscan")

# Report all hits below E-value 10.0 | see docs above for all fields
config = PyHmmscanConfig(evalue_threshold=10.0)

**Config** — `PyHmmscanConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>verbose</code> | <code>int</code> | <code>0</code> | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| <code>device</code> | <code>str</code> | <code>'cpu'</code> | Device to run the tool on (e.g., 'cpu', 'cuda', 'cuda:0', 'cloud') |
| <code>timeout</code> | <code>int &#124; None</code> | <code>3600</code> | Maximum execution time in seconds. None waits indefinitely. |
| <code>seed</code> | <code>int &#124; None</code> | <code>None</code> | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| <code>num_threads</code> | <code>int</code> | <code>0</code> | CPU threads. 0 auto-detects available cores |
| <code>evalue_threshold</code> | <code>float</code> | <code>10.0</code> | Sequence E-value cap. Lower = stricter; 0.001 for confident, 1e-10 for stringent |
| <code>score_threshold</code> | <code>float &#124; None</code> | <code>None</code> | Sequence bit-score floor. Use for cross-DB-size comparisons; overrides E-value if set |
| <code>domain_evalue_threshold</code> | <code>float</code> | <code>10.0</code> | Domain E-value cap. Tighten independently from sequence threshold |
| <code>domain_score_threshold</code> | <code>float &#124; None</code> | <code>None</code> | Domain bit-score floor. For cross-DB-size comparisons; overrides domain E-value |
| <code>inclusion_evalue_threshold</code> | <code>float</code> | <code>0.01</code> | Inclusion E-value cap. Sets 'included' flag; seeds jackhmmer next iteration |
| <code>inclusion_domain_evalue_threshold</code> | <code>float</code> | <code>0.01</code> | Domain inclusion E-value cap. Sets included flag on domain hits |
| <code>z_value</code> | <code>float &#124; None</code> | <code>None</code> | Effective DB size for E-value calc. Set constant for cross-DB compare; None = actual count |
| <code>domain_z_value</code> | <code>float &#124; None</code> | <code>None</code> | Significant hits for domain E-value calc. Set with Z for cross-DB; None = actual |
| <code>skip_filters</code> | <code>bool</code> | <code>False</code> | Disable MSV/Vit/Fwd + bias filters. 10-100x slower but max sensitivity for distant homologs |
| <code>bit_cutoffs</code> | <code>Literal['gathering', 'noise', 'trusted'] &#124; None</code> | <code>None</code> | HMM curated cutoff: 'gathering' (Pfam GA), 'noise' (permissive), 'trusted' (strictest) |

In [11]:
# Run domain scan
hmmscan_result = run_pyhmmer_hmmscan(inputs, config)

Running run_pyhmmer_hmmscan [00:00]

In [12]:
# Display output docs
display_api_reference("pyhmmer", "output", "run_pyhmmer_hmmscan")

# Kinase (seq 0) should be annotated with IPK and Barwin domains
print(f"Found {hmmscan_result.num_sequence_hits} sequence hits")
print(f"Found {hmmscan_result.num_domain_hits} domain hits")

if hmmscan_result.domain_hits:
    print("\nDomain annotations (sequence 0=kinase, 1=hemoglobin):")
    for hit in hmmscan_result.domain_hits:
        print(f"  query={hit.query_name}, domain={hit.target_name}, c_evalue={hit.c_evalue:.2e}, score={hit.domain_score:.1f}")

**Output** — `PyHmmerOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequence_hits</code> | <code>list[SequenceHit]</code> | <code>[]</code> | Sequence-level hits from the search |
| <code>domain_hits</code> | <code>list[DomainHit]</code> | <code>[]</code> | Domain-level hits from the search |

**`SequenceHit`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>query_name</code> | <code>str</code> | required | Name of the query HMM |
| <code>query_accession</code> | <code>str</code> | required | Accession of the query HMM. If not available, set to '-'. |
| <code>query_description</code> | <code>str</code> | required | Description of the query HMM. If not available, set to '-'. |
| <code>query_idx</code> | <code>int</code> | required | Index of the query (0-indexed) |
| <code>target_name</code> | <code>str</code> | required | Name of the target sequence |
| <code>target_accession</code> | <code>str</code> | required | Accession of the target sequence. If not available, set to '-'. |
| <code>target_description</code> | <code>str</code> | required | Description of the target sequence. If not available, set to '-'. |
| <code>evalue</code> | <code>float</code> | required | E-value of the hit |
| <code>score</code> | <code>float</code> | required | Bit score of the full sequence |
| <code>bias</code> | <code>float</code> | required | Bias correction for the sequence score |
| <code>sum_score</code> | <code>float</code> | required | Sum of domain scores |
| <code>reported</code> | <code>bool</code> | required | Whether the hit is reported |
| <code>included</code> | <code>bool</code> | required | Whether the hit is included |
| <code>pvalue</code> | <code>float</code> | required | p-value of the hit |
| <code>num_domains</code> | <code>int</code> | required | Number of domains in the hit |

**`DomainHit`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>query_name</code> | <code>str</code> | required | Name of the query HMM |
| <code>query_accession</code> | <code>str</code> | required | Accession of the query HMM. If not available, set to '-'. |
| <code>query_description</code> | <code>str</code> | required | Description of the query HMM. If not available, set to '-'. |
| <code>query_idx</code> | <code>int</code> | required | Index of the query (0-indexed) |
| <code>target_name</code> | <code>str</code> | required | Name of the target sequence |
| <code>target_accession</code> | <code>str</code> | required | Accession of the target sequence. If not available, set to '-'. |
| <code>target_description</code> | <code>str</code> | required | Description of the target sequence. If not available, set to '-'. |
| <code>hmm_length</code> | <code>int</code> | required | Length of the HMM profile |
| <code>hmm_from</code> | <code>int</code> | required | Start position of the domain match in the HMM (1-indexed) |
| <code>hmm_to</code> | <code>int</code> | required | End position of the domain match in the HMM (1-indexed) |
| <code>target_from</code> | <code>int</code> | required | Start position of the domain match in the target (1-indexed) |
| <code>target_to</code> | <code>int</code> | required | End position of the domain match in the target (1-indexed) |
| <code>target_length</code> | <code>int</code> | required | Length of the target sequence |
| <code>c_evalue</code> | <code>float</code> | required | Conditional E-value of the domain |
| <code>i_evalue</code> | <code>float</code> | required | Independent E-value of the domain |
| <code>domain_score</code> | <code>float</code> | required | Bit score of the domain |
| <code>domain_bias</code> | <code>float</code> | required | Bias correction of the domain score |
| <code>domain_idx</code> | <code>int</code> | required | Index of the domain within the hit (0-indexed) |
| <code>env_from</code> | <code>int</code> | required | Start position of the domain envelope in the target (1-indexed) |
| <code>env_to</code> | <code>int</code> | required | End position of the domain envelope in the target (1-indexed) |
| <code>envelope_score</code> | <code>float</code> | required | Bit score of the domain envelope |
| <code>domain_included</code> | <code>bool</code> | required | Whether the domain passes inclusion thresholds |
| <code>domain_reported</code> | <code>bool</code> | required | Whether the domain passes reporting thresholds |
| <code>domain_pvalue</code> | <code>float</code> | required | P-value of the domain |

Found 2 sequence hits
Found 2 domain hits

Domain annotations (sequence 0=kinase, 1=hemoglobin):
  query=0, domain=IPK, c_evalue=9.61e-50, score=156.2
  query=0, domain=Barwin, c_evalue=3.91e-05, score=10.8


### `run_pyhmmer_phmmer`

`run_pyhmmer_phmmer` performs protein-versus-protein search by building a temporary HMM profile on the fly from the query sequence and scanning it against target sequences. This gives greater sensitivity to distant homologs than a pairwise alignment like BLASTP without requiring a prebuilt profile. It is well-suited for finding orthologs across species or confirming that a designed sequence falls within a known protein family.

In [13]:
from proto_tools import (
    PyPhmmerInput,
    PyPhmmerConfig,
    run_pyhmmer_phmmer,
)

In [14]:
# Display input docs
display_api_reference("pyhmmer", "input", "run_pyhmmer_phmmer")

# Query: human hemoglobin alpha (82 residues)
query_seq = (
    "MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSH"
    "GSAQVKGHGKKVADALTNAVAHVDDMPNALSA"
)

# Three targets at different evolutionary distances
target_seqs = [
    # Mouse hemoglobin alpha: close ortholog, expect strong match
    "MVLSGEDKSNIKAAWGKIGGHGAEYGAEALERMFASFPTTKTYFPHFDVSH"
    "GSAQVKGHGKKVADALASAAGHLDDLPGALSAL",
    # Human hemoglobin beta: same globin family, expect partial match
    "VHLTPEEKSAVTALWGKVNVDEVGGEALGRLLVVYPWTQRFFESFGDLST"
    "PDAVMGNPKVKAHGKKVLGAFSDGLAHLDNLKGT",
    # GFP fragment: unrelated protein, expect no match
    "MSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFICTT"
    "GKLPVPWPTLVTTLTYGVQCFSRYPDHMKQ",
]

phmmer_inputs = PyPhmmerInput(
    sequences=[query_seq],
    target_sequences=target_seqs,
)

**Input** — `PyPhmmerInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequences</code> | <code>list[str]</code> | required | Input sequences as: single sequence string or list of sequence strings |
| <code>target_sequences</code> | <code>list[str]</code> | required | Target sequences as: single sequence string or list of sequence strings |

In [15]:
# Display config docs
display_api_reference("pyhmmer", "config", "run_pyhmmer_phmmer")

# Report all hits below E-value 10.0 | see docs above for all fields
phmmer_config = PyPhmmerConfig(evalue_threshold=10.0)

**Config** — `PyHmmerConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>verbose</code> | <code>int</code> | <code>0</code> | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| <code>device</code> | <code>str</code> | <code>'cpu'</code> | Device to run the tool on (e.g., 'cpu', 'cuda', 'cuda:0', 'cloud') |
| <code>timeout</code> | <code>int &#124; None</code> | <code>3600</code> | Maximum execution time in seconds. None waits indefinitely. |
| <code>seed</code> | <code>int &#124; None</code> | <code>None</code> | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| <code>num_threads</code> | <code>int</code> | <code>0</code> | CPU threads. 0 auto-detects available cores |
| <code>evalue_threshold</code> | <code>float</code> | <code>10.0</code> | Sequence E-value cap. Lower = stricter; 0.001 for confident, 1e-10 for stringent |
| <code>score_threshold</code> | <code>float &#124; None</code> | <code>None</code> | Sequence bit-score floor. Use for cross-DB-size comparisons; overrides E-value if set |
| <code>domain_evalue_threshold</code> | <code>float</code> | <code>10.0</code> | Domain E-value cap. Tighten independently from sequence threshold |
| <code>domain_score_threshold</code> | <code>float &#124; None</code> | <code>None</code> | Domain bit-score floor. For cross-DB-size comparisons; overrides domain E-value |
| <code>inclusion_evalue_threshold</code> | <code>float</code> | <code>0.01</code> | Inclusion E-value cap. Sets 'included' flag; seeds jackhmmer next iteration |
| <code>inclusion_domain_evalue_threshold</code> | <code>float</code> | <code>0.01</code> | Domain inclusion E-value cap. Sets included flag on domain hits |
| <code>z_value</code> | <code>float &#124; None</code> | <code>None</code> | Effective DB size for E-value calc. Set constant for cross-DB compare; None = actual count |
| <code>domain_z_value</code> | <code>float &#124; None</code> | <code>None</code> | Significant hits for domain E-value calc. Set with Z for cross-DB; None = actual |
| <code>skip_filters</code> | <code>bool</code> | <code>False</code> | Disable MSV/Vit/Fwd + bias filters. 10-100x slower but max sensitivity for distant homologs |

In [16]:
# Run phmmer search
phmmer_result = run_pyhmmer_phmmer(phmmer_inputs, phmmer_config)

Running run_pyhmmer_phmmer [00:00]

In [17]:
# Display output docs
display_api_reference("pyhmmer", "output", "run_pyhmmer_phmmer")

# Mouse alpha (0) and human beta (1) should hit; GFP (2) should not
print(f"Found {phmmer_result.num_sequence_hits} sequence hits")
print(f"Found {phmmer_result.num_domain_hits} domain hits")

if phmmer_result.sequence_hits:
    print("\nSequence hits (target 0=mouse alpha, 1=human beta, 2=GFP):")
    for hit in phmmer_result.sequence_hits:
        print(f"  query={hit.query_name}, target={hit.target_name}, evalue={hit.evalue:.2e}, score={hit.score:.1f}")

**Output** — `PyHmmerOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequence_hits</code> | <code>list[SequenceHit]</code> | <code>[]</code> | Sequence-level hits from the search |
| <code>domain_hits</code> | <code>list[DomainHit]</code> | <code>[]</code> | Domain-level hits from the search |

**`SequenceHit`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>query_name</code> | <code>str</code> | required | Name of the query HMM |
| <code>query_accession</code> | <code>str</code> | required | Accession of the query HMM. If not available, set to '-'. |
| <code>query_description</code> | <code>str</code> | required | Description of the query HMM. If not available, set to '-'. |
| <code>query_idx</code> | <code>int</code> | required | Index of the query (0-indexed) |
| <code>target_name</code> | <code>str</code> | required | Name of the target sequence |
| <code>target_accession</code> | <code>str</code> | required | Accession of the target sequence. If not available, set to '-'. |
| <code>target_description</code> | <code>str</code> | required | Description of the target sequence. If not available, set to '-'. |
| <code>evalue</code> | <code>float</code> | required | E-value of the hit |
| <code>score</code> | <code>float</code> | required | Bit score of the full sequence |
| <code>bias</code> | <code>float</code> | required | Bias correction for the sequence score |
| <code>sum_score</code> | <code>float</code> | required | Sum of domain scores |
| <code>reported</code> | <code>bool</code> | required | Whether the hit is reported |
| <code>included</code> | <code>bool</code> | required | Whether the hit is included |
| <code>pvalue</code> | <code>float</code> | required | p-value of the hit |
| <code>num_domains</code> | <code>int</code> | required | Number of domains in the hit |

**`DomainHit`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>query_name</code> | <code>str</code> | required | Name of the query HMM |
| <code>query_accession</code> | <code>str</code> | required | Accession of the query HMM. If not available, set to '-'. |
| <code>query_description</code> | <code>str</code> | required | Description of the query HMM. If not available, set to '-'. |
| <code>query_idx</code> | <code>int</code> | required | Index of the query (0-indexed) |
| <code>target_name</code> | <code>str</code> | required | Name of the target sequence |
| <code>target_accession</code> | <code>str</code> | required | Accession of the target sequence. If not available, set to '-'. |
| <code>target_description</code> | <code>str</code> | required | Description of the target sequence. If not available, set to '-'. |
| <code>hmm_length</code> | <code>int</code> | required | Length of the HMM profile |
| <code>hmm_from</code> | <code>int</code> | required | Start position of the domain match in the HMM (1-indexed) |
| <code>hmm_to</code> | <code>int</code> | required | End position of the domain match in the HMM (1-indexed) |
| <code>target_from</code> | <code>int</code> | required | Start position of the domain match in the target (1-indexed) |
| <code>target_to</code> | <code>int</code> | required | End position of the domain match in the target (1-indexed) |
| <code>target_length</code> | <code>int</code> | required | Length of the target sequence |
| <code>c_evalue</code> | <code>float</code> | required | Conditional E-value of the domain |
| <code>i_evalue</code> | <code>float</code> | required | Independent E-value of the domain |
| <code>domain_score</code> | <code>float</code> | required | Bit score of the domain |
| <code>domain_bias</code> | <code>float</code> | required | Bias correction of the domain score |
| <code>domain_idx</code> | <code>int</code> | required | Index of the domain within the hit (0-indexed) |
| <code>env_from</code> | <code>int</code> | required | Start position of the domain envelope in the target (1-indexed) |
| <code>env_to</code> | <code>int</code> | required | End position of the domain envelope in the target (1-indexed) |
| <code>envelope_score</code> | <code>float</code> | required | Bit score of the domain envelope |
| <code>domain_included</code> | <code>bool</code> | required | Whether the domain passes inclusion thresholds |
| <code>domain_reported</code> | <code>bool</code> | required | Whether the domain passes reporting thresholds |
| <code>domain_pvalue</code> | <code>float</code> | required | P-value of the domain |

Found 2 sequence hits
Found 2 domain hits

Sequence hits (target 0=mouse alpha, 1=human beta, 2=GFP):
  query=0, target=0, evalue=1.62e-47, score=146.9
  query=0, target=1, evalue=1.67e-16, score=48.3


### `run_pyhmmer_nhmmer`

`run_pyhmmer_nhmmer` is the nucleotide counterpart of phmmer. It builds a temporary HMM profile from a DNA or RNA query and scans target nucleotide sequences for homologous regions. This is useful for finding conserved non-coding elements, regulatory regions, or homologous genes at the DNA level where protein-level search is not applicable.

In [18]:
from proto_tools import (
    PyNhmmerInput,
    PyNhmmerConfig,
    run_pyhmmer_nhmmer,
)

In [19]:
# Display input docs
display_api_reference("pyhmmer", "input", "run_pyhmmer_nhmmer")

# Query: human hemoglobin alpha coding DNA (first 150 nt)
query_dna = (
    "ATGGTGCTGTCTCCTGCCGACAAGACCAACGTCAAGGCCGCCTGGGGTAAG"
    "GTCGGCGCGCACGCTGGCGAGTATGGTGCGGAGGCCCTGGAGAGGATGTTC"
    "CTGTCCTTCCCCACCACCAAGACCTACTTCCCGCACTTCGACCTGAGCCAC"
)

# Two targets: mouse hemoglobin alpha DNA (ortholog) and GFP DNA (unrelated)
target_dna = [
    "ATGGTGCTGTCTGAGGACAAGAGCAACATCAAGGCCGCCTGGGGTAAGATC"
    "GGCGGCCACGCTGGCGAGTATGGTGCGGAGGCCCTGGAGAGGATGTTCGCC"
    "TCCTTCCCCACCACCAAGACCTACTTCCCGCACTTCGACGTGAGCCAC",
    "ATGAGTAAAGGAGAAGAACTTTTCACTGGAGTTGTCCCAATTCTTGTTGAAT"
    "TAGATGGTGATGTTAATGGGCACAAATTTTCTGTCAGTGGAGAGGGTGAAGG"
    "TGATGCTACATACGGAAAGCTTACCCTTAAATTTATTTGCACTACTGGAAAA",
]

nhmmer_inputs = PyNhmmerInput(
    sequences=[query_dna],
    target_sequences=target_dna,
)

**Input** — `PyNhmmerInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequences</code> | <code>list[str]</code> | required | Input sequences as: single sequence string or list of sequence strings |
| <code>target_sequences</code> | <code>list[str]</code> | required | Target nucleotide sequences as: single sequence string or list of sequence strings |

In [20]:
# Display config docs
display_api_reference("pyhmmer", "config", "run_pyhmmer_nhmmer")

# Report all hits below E-value 10.0 | see docs above for all fields
nhmmer_config = PyNhmmerConfig(evalue_threshold=10.0)

**Config** — `PyNhmmerConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>verbose</code> | <code>int</code> | <code>0</code> | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| <code>device</code> | <code>str</code> | <code>'cpu'</code> | Device to run the tool on (e.g., 'cpu', 'cuda', 'cuda:0', 'cloud') |
| <code>timeout</code> | <code>int &#124; None</code> | <code>3600</code> | Maximum execution time in seconds. None waits indefinitely. |
| <code>seed</code> | <code>int &#124; None</code> | <code>None</code> | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| <code>num_threads</code> | <code>int</code> | <code>0</code> | CPU threads. 0 auto-detects available cores |
| <code>evalue_threshold</code> | <code>float</code> | <code>10.0</code> | Sequence E-value cap. Lower = stricter; 0.001 for confident, 1e-10 for stringent |
| <code>score_threshold</code> | <code>float &#124; None</code> | <code>None</code> | Sequence bit-score floor. Use for cross-DB-size comparisons; overrides E-value if set |
| <code>domain_evalue_threshold</code> | <code>float</code> | <code>10.0</code> | Domain E-value cap. Tighten independently from sequence threshold |
| <code>domain_score_threshold</code> | <code>float &#124; None</code> | <code>None</code> | Domain bit-score floor. For cross-DB-size comparisons; overrides domain E-value |
| <code>inclusion_evalue_threshold</code> | <code>float</code> | <code>0.01</code> | Inclusion E-value cap. Sets 'included' flag; seeds jackhmmer next iteration |
| <code>inclusion_domain_evalue_threshold</code> | <code>float</code> | <code>0.01</code> | Domain inclusion E-value cap. Sets included flag on domain hits |
| <code>z_value</code> | <code>float &#124; None</code> | <code>None</code> | Effective DB size for E-value calc. Set constant for cross-DB compare; None = actual count |
| <code>domain_z_value</code> | <code>float &#124; None</code> | <code>None</code> | Significant hits for domain E-value calc. Set with Z for cross-DB; None = actual |
| <code>skip_filters</code> | <code>bool</code> | <code>False</code> | Disable MSV/Vit/Fwd + bias filters. 10-100x slower but max sensitivity for distant homologs |
| <code>strand</code> | <code>Literal['both', 'watson', 'crick']</code> | <code>'both'</code> | Strand: 'both' (~2x runtime), 'watson' (forward only), 'crick' (reverse only) |

In [21]:
# Run nhmmer search
nhmmer_result = run_pyhmmer_nhmmer(nhmmer_inputs, nhmmer_config)

Running run_pyhmmer_nhmmer [00:00]

In [22]:
# Display output docs
display_api_reference("pyhmmer", "output", "run_pyhmmer_nhmmer")

# Mouse alpha DNA (0) should hit; GFP DNA (1) should not
print(f"Found {nhmmer_result.num_sequence_hits} sequence hits")
print(f"Found {nhmmer_result.num_domain_hits} domain hits")

if nhmmer_result.sequence_hits:
    print("\nSequence hits (target 0=mouse alpha DNA, 1=GFP DNA):")
    for hit in nhmmer_result.sequence_hits:
        print(f"  query={hit.query_name}, target={hit.target_name}, evalue={hit.evalue:.2e}, score={hit.score:.1f}")

**Output** — `PyHmmerOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequence_hits</code> | <code>list[SequenceHit]</code> | <code>[]</code> | Sequence-level hits from the search |
| <code>domain_hits</code> | <code>list[DomainHit]</code> | <code>[]</code> | Domain-level hits from the search |

**`SequenceHit`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>query_name</code> | <code>str</code> | required | Name of the query HMM |
| <code>query_accession</code> | <code>str</code> | required | Accession of the query HMM. If not available, set to '-'. |
| <code>query_description</code> | <code>str</code> | required | Description of the query HMM. If not available, set to '-'. |
| <code>query_idx</code> | <code>int</code> | required | Index of the query (0-indexed) |
| <code>target_name</code> | <code>str</code> | required | Name of the target sequence |
| <code>target_accession</code> | <code>str</code> | required | Accession of the target sequence. If not available, set to '-'. |
| <code>target_description</code> | <code>str</code> | required | Description of the target sequence. If not available, set to '-'. |
| <code>evalue</code> | <code>float</code> | required | E-value of the hit |
| <code>score</code> | <code>float</code> | required | Bit score of the full sequence |
| <code>bias</code> | <code>float</code> | required | Bias correction for the sequence score |
| <code>sum_score</code> | <code>float</code> | required | Sum of domain scores |
| <code>reported</code> | <code>bool</code> | required | Whether the hit is reported |
| <code>included</code> | <code>bool</code> | required | Whether the hit is included |
| <code>pvalue</code> | <code>float</code> | required | p-value of the hit |
| <code>num_domains</code> | <code>int</code> | required | Number of domains in the hit |

**`DomainHit`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>query_name</code> | <code>str</code> | required | Name of the query HMM |
| <code>query_accession</code> | <code>str</code> | required | Accession of the query HMM. If not available, set to '-'. |
| <code>query_description</code> | <code>str</code> | required | Description of the query HMM. If not available, set to '-'. |
| <code>query_idx</code> | <code>int</code> | required | Index of the query (0-indexed) |
| <code>target_name</code> | <code>str</code> | required | Name of the target sequence |
| <code>target_accession</code> | <code>str</code> | required | Accession of the target sequence. If not available, set to '-'. |
| <code>target_description</code> | <code>str</code> | required | Description of the target sequence. If not available, set to '-'. |
| <code>hmm_length</code> | <code>int</code> | required | Length of the HMM profile |
| <code>hmm_from</code> | <code>int</code> | required | Start position of the domain match in the HMM (1-indexed) |
| <code>hmm_to</code> | <code>int</code> | required | End position of the domain match in the HMM (1-indexed) |
| <code>target_from</code> | <code>int</code> | required | Start position of the domain match in the target (1-indexed) |
| <code>target_to</code> | <code>int</code> | required | End position of the domain match in the target (1-indexed) |
| <code>target_length</code> | <code>int</code> | required | Length of the target sequence |
| <code>c_evalue</code> | <code>float</code> | required | Conditional E-value of the domain |
| <code>i_evalue</code> | <code>float</code> | required | Independent E-value of the domain |
| <code>domain_score</code> | <code>float</code> | required | Bit score of the domain |
| <code>domain_bias</code> | <code>float</code> | required | Bias correction of the domain score |
| <code>domain_idx</code> | <code>int</code> | required | Index of the domain within the hit (0-indexed) |
| <code>env_from</code> | <code>int</code> | required | Start position of the domain envelope in the target (1-indexed) |
| <code>env_to</code> | <code>int</code> | required | End position of the domain envelope in the target (1-indexed) |
| <code>envelope_score</code> | <code>float</code> | required | Bit score of the domain envelope |
| <code>domain_included</code> | <code>bool</code> | required | Whether the domain passes inclusion thresholds |
| <code>domain_reported</code> | <code>bool</code> | required | Whether the domain passes reporting thresholds |
| <code>domain_pvalue</code> | <code>float</code> | required | P-value of the domain |

Found 1 sequence hits
Found 1 domain hits

Sequence hits (target 0=mouse alpha DNA, 1=GFP DNA):
  query=0, target=0, evalue=1.27e-36, score=112.8


### `run_pyhmmer_jackhmmer`

`run_pyhmmer_jackhmmer` performs iterative protein sequence search. In the first round it works like phmmer, building a profile from the query and finding initial hits. It then incorporates those hits into a refined profile and searches again, repeating until convergence or the maximum number of iterations. This iterative refinement allows jackhmmer to detect very distant homologs that single-pass methods would miss.

In [23]:
from proto_tools import (
    PyJackhmmerInput,
    PyJackhmmerConfig,
    run_pyhmmer_jackhmmer,
)

In [24]:
# Display input docs
display_api_reference("pyhmmer", "input", "run_pyhmmer_jackhmmer")

# Reuse the hemoglobin query and targets from the phmmer example
jackhmmer_inputs = PyJackhmmerInput(
    sequences=[query_seq],
    target_sequences=target_seqs,
)

**Input** — `PyJackhmmerInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequences</code> | <code>list[str]</code> | required | Input sequences as: single sequence string or list of sequence strings |
| <code>target_sequences</code> | <code>list[str]</code> | required | Target sequences as: single sequence string or list of sequence strings |

In [25]:
# Display config docs
display_api_reference("pyhmmer", "config", "run_pyhmmer_jackhmmer")

# Three iterations to refine the profile | see docs above for all fields
jackhmmer_config = PyJackhmmerConfig(
    max_iterations=3,
    evalue_threshold=10.0,
)

**Config** — `PyJackhmmerConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>verbose</code> | <code>int</code> | <code>0</code> | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| <code>device</code> | <code>str</code> | <code>'cpu'</code> | Device to run the tool on (e.g., 'cpu', 'cuda', 'cuda:0', 'cloud') |
| <code>timeout</code> | <code>int &#124; None</code> | <code>3600</code> | Maximum execution time in seconds. None waits indefinitely. |
| <code>seed</code> | <code>int &#124; None</code> | <code>None</code> | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| <code>num_threads</code> | <code>int</code> | <code>0</code> | CPU threads. 0 auto-detects available cores |
| <code>evalue_threshold</code> | <code>float</code> | <code>10.0</code> | Sequence E-value cap. Lower = stricter; 0.001 for confident, 1e-10 for stringent |
| <code>score_threshold</code> | <code>float &#124; None</code> | <code>None</code> | Sequence bit-score floor. Use for cross-DB-size comparisons; overrides E-value if set |
| <code>domain_evalue_threshold</code> | <code>float</code> | <code>10.0</code> | Domain E-value cap. Tighten independently from sequence threshold |
| <code>domain_score_threshold</code> | <code>float &#124; None</code> | <code>None</code> | Domain bit-score floor. For cross-DB-size comparisons; overrides domain E-value |
| <code>inclusion_evalue_threshold</code> | <code>float</code> | <code>0.01</code> | Inclusion E-value cap. Sets 'included' flag; seeds jackhmmer next iteration |
| <code>inclusion_domain_evalue_threshold</code> | <code>float</code> | <code>0.01</code> | Domain inclusion E-value cap. Sets included flag on domain hits |
| <code>z_value</code> | <code>float &#124; None</code> | <code>None</code> | Effective DB size for E-value calc. Set constant for cross-DB compare; None = actual count |
| <code>domain_z_value</code> | <code>float &#124; None</code> | <code>None</code> | Significant hits for domain E-value calc. Set with Z for cross-DB; None = actual |
| <code>skip_filters</code> | <code>bool</code> | <code>False</code> | Disable MSV/Vit/Fwd + bias filters. 10-100x slower but max sensitivity for distant homologs |
| <code>max_iterations</code> | <code>int</code> | <code>5</code> | Max jackhmmer iterations; raise to 8-10 for distant homologs, 1-2 for fast preview |

In [26]:
# Run iterative jackhmmer search
jackhmmer_result = run_pyhmmer_jackhmmer(jackhmmer_inputs, jackhmmer_config)

Running run_pyhmmer_jackhmmer [00:00]

In [27]:
# Display output docs
display_api_reference("pyhmmer", "output", "run_pyhmmer_jackhmmer")

# Iterative refinement often finds the beta globin hit at much higher score than phmmer
print(f"Found {jackhmmer_result.num_sequence_hits} sequence hits")
print(f"Found {jackhmmer_result.num_domain_hits} domain hits")

if jackhmmer_result.sequence_hits:
    print("\nSequence hits:")
    for hit in jackhmmer_result.sequence_hits:
        print(f"  query={hit.query_name}, target={hit.target_name}, evalue={hit.evalue:.2e}, score={hit.score:.1f}")

**Output** — `PyHmmerOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequence_hits</code> | <code>list[SequenceHit]</code> | <code>[]</code> | Sequence-level hits from the search |
| <code>domain_hits</code> | <code>list[DomainHit]</code> | <code>[]</code> | Domain-level hits from the search |

**`SequenceHit`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>query_name</code> | <code>str</code> | required | Name of the query HMM |
| <code>query_accession</code> | <code>str</code> | required | Accession of the query HMM. If not available, set to '-'. |
| <code>query_description</code> | <code>str</code> | required | Description of the query HMM. If not available, set to '-'. |
| <code>query_idx</code> | <code>int</code> | required | Index of the query (0-indexed) |
| <code>target_name</code> | <code>str</code> | required | Name of the target sequence |
| <code>target_accession</code> | <code>str</code> | required | Accession of the target sequence. If not available, set to '-'. |
| <code>target_description</code> | <code>str</code> | required | Description of the target sequence. If not available, set to '-'. |
| <code>evalue</code> | <code>float</code> | required | E-value of the hit |
| <code>score</code> | <code>float</code> | required | Bit score of the full sequence |
| <code>bias</code> | <code>float</code> | required | Bias correction for the sequence score |
| <code>sum_score</code> | <code>float</code> | required | Sum of domain scores |
| <code>reported</code> | <code>bool</code> | required | Whether the hit is reported |
| <code>included</code> | <code>bool</code> | required | Whether the hit is included |
| <code>pvalue</code> | <code>float</code> | required | p-value of the hit |
| <code>num_domains</code> | <code>int</code> | required | Number of domains in the hit |

**`DomainHit`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>query_name</code> | <code>str</code> | required | Name of the query HMM |
| <code>query_accession</code> | <code>str</code> | required | Accession of the query HMM. If not available, set to '-'. |
| <code>query_description</code> | <code>str</code> | required | Description of the query HMM. If not available, set to '-'. |
| <code>query_idx</code> | <code>int</code> | required | Index of the query (0-indexed) |
| <code>target_name</code> | <code>str</code> | required | Name of the target sequence |
| <code>target_accession</code> | <code>str</code> | required | Accession of the target sequence. If not available, set to '-'. |
| <code>target_description</code> | <code>str</code> | required | Description of the target sequence. If not available, set to '-'. |
| <code>hmm_length</code> | <code>int</code> | required | Length of the HMM profile |
| <code>hmm_from</code> | <code>int</code> | required | Start position of the domain match in the HMM (1-indexed) |
| <code>hmm_to</code> | <code>int</code> | required | End position of the domain match in the HMM (1-indexed) |
| <code>target_from</code> | <code>int</code> | required | Start position of the domain match in the target (1-indexed) |
| <code>target_to</code> | <code>int</code> | required | End position of the domain match in the target (1-indexed) |
| <code>target_length</code> | <code>int</code> | required | Length of the target sequence |
| <code>c_evalue</code> | <code>float</code> | required | Conditional E-value of the domain |
| <code>i_evalue</code> | <code>float</code> | required | Independent E-value of the domain |
| <code>domain_score</code> | <code>float</code> | required | Bit score of the domain |
| <code>domain_bias</code> | <code>float</code> | required | Bias correction of the domain score |
| <code>domain_idx</code> | <code>int</code> | required | Index of the domain within the hit (0-indexed) |
| <code>env_from</code> | <code>int</code> | required | Start position of the domain envelope in the target (1-indexed) |
| <code>env_to</code> | <code>int</code> | required | End position of the domain envelope in the target (1-indexed) |
| <code>envelope_score</code> | <code>float</code> | required | Bit score of the domain envelope |
| <code>domain_included</code> | <code>bool</code> | required | Whether the domain passes inclusion thresholds |
| <code>domain_reported</code> | <code>bool</code> | required | Whether the domain passes reporting thresholds |
| <code>domain_pvalue</code> | <code>float</code> | required | P-value of the domain |

Found 2 sequence hits
Found 2 domain hits

Sequence hits:
  query=0, target=0, evalue=1.21e-53, score=167.1
  query=0, target=1, evalue=1.80e-42, score=131.3


## Export Results

Search results from any PyHMMER function can be exported to CSV or JSON. CSV produces one row per hit with alignment statistics; JSON preserves the full nested structure including domain-level hits.

In [28]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

jackhmmer_result.export(name="pyhmmer_jackhmmer_results", export_path=output_dir, file_format="csv")
print(f"Results exported to {output_dir}")

Results exported to example_output
